In [0]:
%sql
SHOW CATALOGS

In [0]:
spark.sql("USE CATALOG tp7_databricks_spark")
spark.sql("USE SCHEMA iris_species")

print(f"Catálogo actual: {spark.catalog.currentCatalog()}")
print(f"Esquema actual:  {spark.catalog.currentDatabase()}")

In [0]:
import os
import shutil

catalog = spark.catalog.currentCatalog()
schema = spark.catalog.currentDatabase()
target_file = f"/Volumes/{catalog}/{schema}/raw_data/iris.csv"

# 1. Crear volumen si no existe
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{catalog}`.`{schema}`.raw_data")

# 2. Copiar el csv de data/ al volumen si no fue copiado antes
if not os.path.exists(target_file):
    shutil.copyfile("../data/iris.csv", target_file)

# 3. Cargar DataFrame
df_iris = spark.read.csv(target_file, header=True, inferSchema=True)
display(df_iris.limit(5))

In [0]:
%sql
CREATE TABLE IF NOT EXISTS iris_bronze (
  sepal_length DOUBLE,
  sepal_width DOUBLE,
  petal_length DOUBLE,
  petal_width DOUBLE,
  species STRING
);

In [0]:
# Load data incrementaly with COPY INTO
result = spark.sql(f"""
    COPY INTO iris_bronze
    FROM '/Volumes/{catalog}/{schema}/raw_data/'
    FILEFORMAT = CSV
    FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')
""")
result.display()

In [0]:
%sql
SELECT * FROM iris_bronze;

In [0]:
%sql
CREATE OR REPLACE TABLE iris_silver AS
SELECT
  -- 1. Identificador único por fila
  monotonically_increasing_id() AS iris_id,

  -- 2. Columnas originales
  sepal_length,
  sepal_width,
  petal_length,
  petal_width,
  species,

  -- 3. Columnas calculadas (Feature Engineering)
  ROUND(sepal_length / sepal_width, 2) AS sepal_ratio,
  ROUND(petal_length / petal_width, 2) AS petal_ratio,
  ROUND(petal_length * petal_width, 2) AS petal_area,

  -- 4. Columnas de metadata
  current_timestamp() AS processed_timestamp

FROM (
  -- Subconsulta para eliminar duplicados exactos antes de asignar IDs
  SELECT DISTINCT
    sepal_length,
    sepal_width,
    petal_length,
    petal_width,
    species
  FROM iris_bronze
  WHERE 
    sepal_length > 0 AND sepal_length IS NOT NULL
    AND sepal_width > 0 AND sepal_width IS NOT NULL
    AND petal_length > 0 AND petal_length IS NOT NULL
    AND petal_width > 0 AND petal_width IS NOT NULL
);

In [0]:
%sql
SELECT * FROM iris_silver;

In [0]:
%sql
-- 1. Vista temporal con agregaciones estadísticas completas
CREATE OR REPLACE TEMP VIEW temp_species_metrics AS
SELECT
  species,
  COUNT(*) AS sample_count,
  
  -- Sépalo
  ROUND(AVG(sepal_length), 2) AS avg_sepal_length,
  ROUND(STDDEV(sepal_length), 2) AS std_sepal_length,
  ROUND(MIN(sepal_length), 2) AS min_sepal_length,
  ROUND(MAX(sepal_length), 2) AS max_sepal_length,

  -- Pétalo
  ROUND(AVG(petal_length), 2) AS avg_petal_length,
  ROUND(STDDEV(petal_length), 2) AS std_petal_length,
  ROUND(MIN(petal_length), 2) AS min_petal_length,
  ROUND(MAX(petal_length), 2) AS max_petal_length,

  -- Métricas derivadas en Silver
  ROUND(AVG(petal_area), 2) AS avg_petal_area,
  ROUND(STDDEV(petal_area), 2) AS std_petal_area,
  ROUND(AVG(sepal_ratio), 2) AS avg_sepal_ratio,

  current_timestamp() AS updated_at
FROM iris_silver
GROUP BY species;

-- 2. Creación formal de la tabla en Gold con tipado explícito
CREATE TABLE IF NOT EXISTS iris_species_metrics_gold (
  species STRING,
  sample_count BIGINT,
  avg_sepal_length DOUBLE,
  std_sepal_length DOUBLE,
  min_sepal_length DOUBLE,
  max_sepal_length DOUBLE,
  avg_petal_length DOUBLE,
  std_petal_length DOUBLE,
  min_petal_length DOUBLE,
  max_petal_length DOUBLE,
  avg_petal_area DOUBLE,
  std_petal_area DOUBLE,
  avg_sepal_ratio DOUBLE,
  updated_at TIMESTAMP
);

-- 3. Inserción idempotente
INSERT OVERWRITE iris_species_metrics_gold
SELECT * FROM temp_species_metrics;

In [0]:
%sql
SELECT * FROM iris_species_metrics_gold;

In [0]:
%sql
-- 1. Cálculo de percentiles mediante aproximación distribuida
CREATE OR REPLACE TEMP VIEW temp_species_thresholds AS
SELECT
  species,
  ROUND(percentile_approx(petal_length, 0.25), 2) AS p25_petal_length,
  ROUND(percentile_approx(petal_length, 0.50), 2) AS median_petal_length,
  ROUND(percentile_approx(petal_length, 0.75), 2) AS p75_petal_length,
  ROUND(percentile_approx(petal_width, 0.25), 2) AS p25_petal_width,
  ROUND(percentile_approx(petal_width, 0.50), 2) AS median_petal_width,
  ROUND(percentile_approx(petal_width, 0.75), 2) AS p75_petal_width,
  ROUND(percentile_approx(petal_area, 0.50), 2) AS median_petal_area,
  current_timestamp() AS updated_at
FROM iris_silver
GROUP BY species;

-- 2. DDL formal
CREATE TABLE IF NOT EXISTS iris_thresholds_gold (
  species STRING,
  p25_petal_length DOUBLE,
  median_petal_length DOUBLE,
  p75_petal_length DOUBLE,
  p25_petal_width DOUBLE,
  median_petal_width DOUBLE,
  p75_petal_width DOUBLE,
  median_petal_area DOUBLE,
  updated_at TIMESTAMP
);

-- 3. Carga de datos
INSERT OVERWRITE iris_thresholds_gold
SELECT * FROM temp_species_thresholds;

In [0]:
%sql
SELECT * FROM iris_thresholds_gold;

In [0]:
%sql
-- 1. Bins categóricos por rango de tamaño de flor
CREATE OR REPLACE TEMP VIEW temp_distribution_bins AS
SELECT
  species,
  CASE 
    WHEN petal_length < 2.5 THEN 'pequeno (<2.5cm)'
    WHEN petal_length BETWEEN 2.5 AND 4.8 THEN 'mediano (2.5-4.8cm)'
    ELSE 'grande (>4.8cm)'
  END AS petal_size_group,
  COUNT(*) AS total_count,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY species), 2) AS percentage_within_species,
  current_timestamp() AS updated_at
FROM iris_silver
GROUP BY 
  species,
  CASE 
    WHEN petal_length < 2.5 THEN 'pequeno (<2.5cm)'
    WHEN petal_length BETWEEN 2.5 AND 4.8 THEN 'mediano (2.5-4.8cm)'
    ELSE 'grande (>4.8cm)'
  END;

-- 2. DDL formal
CREATE TABLE IF NOT EXISTS iris_distribution_bins_gold (
  species STRING,
  petal_size_group STRING,
  total_count BIGINT,
  percentage_within_species DOUBLE,
  updated_at TIMESTAMP
);

-- 3. Carga
INSERT OVERWRITE iris_distribution_bins_gold
SELECT * FROM temp_distribution_bins;

In [0]:
%sql
SELECT * FROM iris_distribution_bins_gold;

Objetivo del análisis: Evaluar la magnitud media y el grado de dispersión del área de los pétalos entre las distintas clases biológicas.

Interpretación técnica: La especie Iris setosa exhibe una morfología compacta y una dispersión mínima, lo que permite su aislamiento lineal frente a las demás clases. Por su parte, Iris versicolor e Iris virginica presentan medias escalonadas pero con intervalos de desviación contiguos, justificando el uso de fronteras de decisión no lineales en etapas de modelado predictivo.

In [0]:
import matplotlib.pyplot as plt

df_metrics = spark.table("iris_species_metrics_gold").toPandas()

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(
    df_metrics["species"],
    df_metrics["avg_petal_area"],
    yerr=df_metrics["std_petal_area"],
    capsize=5,
    color=["#4c72b0", "#dd8452", "#55a868"]
)

ax.set_title("Área Media de Pétalo por Especie (± Desviación Estándar)")
ax.set_xlabel("Especie")
ax.set_ylabel("Área Promedio (cm²)")
ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

Objetivo del análisis: Determinar la solidez de los umbrales de partición univariados mediante percentiles robustos frente a valores atípicos (outliers).
Interpretación técnica: La mediana y el rango intercuartílico ($p25 - p75$) de la longitud del pétalo confirman la ausencia de solapamiento entre setosa ($< 2$ cm) y el grupo restante. La brecha entre el $p75$ de versicolor y el $p25$ de virginica establece los puntos de corte óptimos para algoritmos basados en árboles de decisión (Decision Trees o Random Forest).

In [0]:
import matplotlib.pyplot as plt

df_thresh = spark.table("iris_thresholds_gold").toPandas()

fig, ax = plt.subplots(figsize=(6, 4))

for idx, row in df_thresh.iterrows():
    # Línea vertical para el rango p25 - p75
    ax.vlines(
        x=row["species"],
        ymin=row["p25_petal_length"],
        ymax=row["p75_petal_length"],
        color="#2b5c8f",
        linewidth=5,
        label="Rango IQR (p25-p75)" if idx == 0 else ""
    )
    # Diamante para la mediana (p50)
    ax.scatter(
        row["species"],
        row["median_petal_length"],
        color="#c44e52",
        marker="D",
        s=60,
        zorder=3,
        label="Mediana" if idx == 0 else ""
    )

ax.set_title("Rango Intercuartílico (p25 - p75) y Mediana de Longitud de Pétalo")
ax.set_xlabel("Especie")
ax.set_ylabel("Longitud de Pétalo (cm)")
ax.legend(loc="upper left")
ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

Objetivo del análisis: Cuantificar la homogeneidad intraclase mediante la discretización de variables continuas en categorías de tamaño.

Interpretación técnica: El 100% de los especímenes de Iris setosa se concentran en la categoría pequeno (<2.5cm). En contraste, Iris versicolor se distribuye predominantemente en mediano con apariciones marginales en grande, mientras que Iris virginica domina la categoría superior, permitiendo validar heurísticas operativas y reglas de negocio sin requerir inferencia probabilística.

In [0]:
import matplotlib.pyplot as plt

df_bins = spark.table("iris_distribution_bins_gold").toPandas()

# Pivotear para crear barras apiladas al 100%
pivot_df = df_bins.pivot(
    index="species",
    columns="petal_size_group",
    values="percentage_within_species"
).fillna(0)

ax = pivot_df.plot(kind="bar", stacked=True, figsize=(7, 4), colormap="viridis")

ax.set_title("Distribución Proporcional de Clases Morfológicas por Especie")
ax.set_xlabel("Especie")
ax.set_ylabel("Proporción (%)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title="Categoría", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()